# 08. Batch Workflow and Reporting

This notebook demonstrates a config-driven batch run where each site is processed through **BestFit model operations** and exported to tables.

In [1]:
import pythonnet
pythonnet.load("coreclr")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_dotnet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from pathlib import Path
from RMC.BestFit import ExactData
from RMC.BestFit.Models import DataFrame, UnivariateDistribution
from Numerics.Distributions import UnivariateDistributionType

In [ ]:
config = {
    'sites': ['A101', 'A102', 'A103', 'A104'],
    'n_years': 40,
    'seed': 77,
    'dist_type': 'GEV',
}
config

In [ ]:
rng = np.random.default_rng(config['seed'])
rows = []

for i, site in enumerate(config['sites']):
    years = np.arange(1980, 1980 + config['n_years'])
    base = rng.uniform(7500, 12500)
    peaks = np.maximum(500, base + 30*(years-years.min()) + rng.normal(0, 850, len(years)))

    # Build BestFit DataFrame per site
    df = DataFrame()
    for y, q in zip(years, peaks):
        df.ExactSeries.Add(ExactData(int(y), float(q)))

    # BestFit univariate model per site
    m = UnivariateDistribution(df, UnivariateDistributionType.GeneralizedExtremeValue)
    mu = float(np.mean(peaks))
    sd = float(np.std(peaks, ddof=1))
    p = convert_to_dotnet_array([mu, max(1e-6, 0.8*sd), 0.08])
    m.SetParameterValues(p)

    ll = float(m.LogLikelihood(p))
    q10 = float(m.Distribution.InverseCDF(0.90))
    q50 = float(m.Distribution.InverseCDF(0.98))
    q100 = float(m.Distribution.InverseCDF(0.99))

    rows.append({
        'site': site,
        'n_years': len(peaks),
        'mean_peak': mu,
        'std_peak': sd,
        'log_likelihood': ll,
        'q10': q10,
        'q50': q50,
        'q100': q100,
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
out_dir = Path('../outputs/tables')
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'bestfit_batch_summary.csv', index=False)
summary.to_json(out_dir / 'bestfit_batch_summary.json', orient='records', indent=2)
print('Wrote', out_dir / 'bestfit_batch_summary.csv')
print('Wrote', out_dir / 'bestfit_batch_summary.json')

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(summary['site'], summary['q100'])
plt.ylabel('Q100')
plt.xlabel('Site')
plt.title('Batch BestFit Q100 by Site')
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()

## Summary and Exercises

You ran a multi-site batch workflow based on BestFit `DataFrame` + `UnivariateDistribution` operations and exported standardized outputs.

Recommended exercises:
1. Add distribution choice to config and compare batch summaries by model family.
2. Add validation checks for minimum record length before fitting.
3. Export a second table with return-period curves per site.